# 🎙️ Voice Studio Pro — Google Colab (Free GPU)
**Tez · Unlimited characters · Voice Clone · 28 Premium Voices** — aap ke PC par koi load nahi.

### Kaise chalayein
1. Upar menu mein **Runtime → Change runtime type → T4 GPU → Save**
2. Neeche wale cell ke bayen taraf **▶ play button** dabayein
3. Google Drive ki ijazat maange to **Allow** karein (taake cloned voices save rahein)
4. 3–6 minute baad neeche **`https://xxxx.gradio.live`** link aayega — use kholein. Bas!

⚠️ Ye window **khuli rakhein**. Band karne par tool band ho jayega. Agle din dobara ▶ dabayein — cloned voices Drive mein save rahengi.

In [ ]:
#@title ▶ Voice Studio Pro chalayein
save_voices_to_drive = True #@param {type:"boolean"}
import os, subprocess, sys

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

def works(mod):
    r = sh(f'python -c "import {mod}"')
    return r.returncode == 0, (r.stderr.strip().splitlines() or [""])[-1]

print("Python", sys.version.split()[0])
gpu = sh("nvidia-smi -L").stdout.strip()
print("✅ GPU mil gaya:", gpu) if gpu else print("⚠️ GPU on nahi hai! Runtime > Change runtime type > T4 GPU > Save, phir ▶ dobara dabayein.\n")

voices_dir = "/content/my_voices"
if save_voices_to_drive:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        voices_dir = "/content/drive/MyDrive/VoiceStudioPro/my_voices"
    except Exception as e:
        print("Drive connect nahi hua, voices sirf is session tak rahengi:", e)
os.makedirs(voices_dir, exist_ok=True)
os.makedirs("/content/vs", exist_ok=True)

SPACY = "https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl"

# ---- 1. Premium voices (Kokoro) ----
ok, err = works("kokoro")
if not ok:
    print("⏳ [1/2] Premium voices install ho rahi hain...")
    r = sh(f'pip install -q kokoro==0.9.4 gradio==6.8.0 lameenc soundfile librosa "setuptools<81" {SPACY}')
    ok, err = works("kokoro")
    print("✅ Premium voices tayyar" if ok else "❌ Kokoro install nahi hua:\n" + r.stderr[-1500:])

# ---- 2. Voice clone (Chatterbox) ----
ok, err = works("chatterbox.tts_turbo")
if not ok:
    print("⏳ [2/2] Voice clone engine install ho raha hai (3–5 minute)...")
    r = sh('pip install -q chatterbox-tts==0.1.7 "setuptools<81"')
    ok, err = works("chatterbox.tts_turbo")
    if not ok:
        print("   Pehla tareeqa kaam nahi kiya, doosra tareeqa try kar raha hoon...")
        sh('pip install -q --no-deps chatterbox-tts==0.1.7')
        sh('pip install -q s3tokenizer resemble-perth conformer==0.3.2 "diffusers==0.29.0" pyloudnorm omegaconf '
           'spacy-pkuseg pykakasi==2.3.0 librosa safetensors "setuptools<81"')
        ok, err = works("chatterbox.tts_turbo")
    if ok:
        print("✅ Voice clone engine tayyar")
    else:
        print("❌ Voice clone engine install nahi hua. Is ka screenshot bhejein:\n   " + err + "\n" + r.stderr[-1200:])

subprocess.run("wget -q -O /content/vs/app.py https://raw.githubusercontent.com/ammarhussan555-tech/cutkaar/main/voice-studio-pro/app.py", shell=True, check=True)
os.environ.update(VS_SHARE="1", VS_NOBROWSER="1", VS_VOICES_DIR=voices_dir, HF_HOME="/content/hf")
print("\n🚀 Tool shuru ho raha hai — neeche 'public URL' wala https://....gradio.live link kholein")
print("   (Upar 'Voice Studio Pro' likha ho aur '🧬 Voice Clone' tab dikhe to clone engine chal raha hai)\n")
!cd /content/vs && python app.py
